# Lab - BUILD: A Batch Pipeline That Survives the Inbox

Three tasks that put the checked extraction to work on a folder of emails. One working function is a demo; a batch that survives a bad file, gives every record a status and a reason, and writes its output safely is something a team can rely on.

The setup cell writes the support inbox export - six text files - into an `emails` folder.

| Task | What you learn |
| --- | --- |
| 1 | Error isolation: one unreadable file gets an error record, the rest go on |
| 2 | A status and a reason for every record - ok, needs review, error |
| 3 | Write the output atomically, and summarize the batch by status |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re
from collections import Counter
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# --- The support inbox export: one text file per email (provided) ---
EMAILS = {
    "001.txt": "Hi, I want to return the blue jacket from order ORD-12345678.\n"
               "It arrived last week but it is too small. Thanks, Alex",
    "002.txt": "Hello, the shoes I got are damaged. Please tell me how to send\n"
               "them back. Sam",
    "003.txt": "Order ORD-87654321: the backpack is bigger than I expected and\n"
               "I would like to return it. Best, Priya",
    "004.txt": "The headphones from order ORD-7788 stopped working after two\n"
               "days. I want a return. Kim",
    "005.txt": "I ordered a lamp (ORD-11112222) as a gift but the recipient\n"
               "already has one. Can I send it back? Jordan",
}
Path("emails").mkdir(exist_ok=True)
for name, text in EMAILS.items():
    (Path("emails") / name).write_text(text, encoding="utf-8")
# A sixth export the inbox tool wrote in another encoding - it happens.
(Path("emails") / "006.txt").write_bytes("Retour für Bestellung ".encode("latin-1") + b"\xff\xfe")

# --- The checked extraction from the validation lab (provided) ---
INSTRUCTION = ("Extract the return request from this customer email. Order IDs look like "
               "ORD-12345678. If the email gives no order number, set order_id to null. "
               "Never invent one.")
SCHEMA = {
    "type": "object",
    "properties": {
        "order_id": {"type": ["string", "null"]},
        "item": {"type": "string"},
        "reason": {"type": "string", "enum": ["too_small", "too_big", "damaged", "other"]},
    },
    "required": ["order_id", "item", "reason"],
    "additionalProperties": False,
}


def validate(data):
    order_id = data["order_id"]
    if order_id is not None and not re.fullmatch(r"ORD-\d{8}", order_id):
        return f"order_id {order_id!r} is not a valid order ID"


def extract(email):
    messages = [{"role": "user", "content": INSTRUCTION + "\n\n" + email}]
    for attempt in range(3):
        response = client.messages.create(
            model=MODEL, max_tokens=1000,
            output_config={"effort": "low",
                           "format": {"type": "json_schema", "schema": SCHEMA}},
            messages=messages)
        text = next(b.text for b in response.content if b.type == "text")
        data = json.loads(text)
        error = validate(data)
        if error is None:
            return data
        messages += [{"role": "assistant", "content": text},
                     {"role": "user", "content": error + ". Fix it, or use null."}]
    return None

---

## Task 1 - One bad file must not stop the batch

In [ ]:
# ============================================================
# TASK 1 - One bad file must not stop the batch
# ============================================================
#
# WHAT TO DO
#   Walk through the files in the folder, in order. Read each
#   one as UTF-8 and extract the fields. When a file cannot be
#   read, record the error for that file and go on with the
#   next one.
#
# WHY IT MATTERS
#   One working function is a demo; a batch the support team can
#   use has to survive the inbox. Exports come in odd encodings,
#   files get truncated, a request fails. Without a try around
#   each file, the sixth file throws away the work on the first
#   five. With it, the bad file gets a record that says what
#   went wrong, and the batch finishes.
#
# WHERE TO SEE IT IN THE LECTURE
#   "We walk through the files in the folder, in order".
#
# HOW TO DO IT
#   Three blanks:
#
#       "*.txt"              the files to read
#       "utf-8"              the encoding the inbox promises
#       UnicodeDecodeError   the error a wrongly encoded file raises
# ============================================================

requests = []
try:
    # BEGIN SOLUTION
    for path in sorted(Path("emails").glob("*.txt")):
        try:
            data = extract(path.read_text(encoding="utf-8"))
        except UnicodeDecodeError as exc:
            requests.append({"email": path.name, "error": f"cannot read file: {exc.reason}"})
            continue
        data["email"] = path.name
        requests.append(data)
    # SCAFFOLD: for path in sorted(Path("emails").glob(...)):   # "*.txt"
    # SCAFFOLD:     try:
    # SCAFFOLD:         data = extract(path.read_text(encoding=...))   # "utf-8"
    # SCAFFOLD:     except ... as exc:   # UnicodeDecodeError
    # SCAFFOLD:         requests.append({"email": path.name, "error": f"cannot read file: {exc.reason}"})
    # SCAFFOLD:         continue
    # SCAFFOLD:     data["email"] = path.name
    # SCAFFOLD:     requests.append(data)
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for row in requests:
    print(row["email"], row.get("order_id"), row.get("error", ""))

check("isolate_errors", requests=requests)

---

## Task 2 - A status and a reason for every record

In [ ]:
# ============================================================
# TASK 2 - A status and a reason for every record
# ============================================================
#
# WHAT TO DO
#   Give every record a status and, when it is not ok, a reason:
#   "error" with the read error, "needs review" when there is no
#   order ID, "ok" otherwise. Print one line per record.
#
# WHY IT MATTERS
#   The status routes the work: ok records go to the order
#   system, the rest to a person. The reason saves that person
#   from opening every file to find out what is wrong. And a
#   missing order number stays missing: pushing Claude until it
#   produces one would produce a wrong one.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then we add two fields of our own".
#
# HOW TO DO IT
#   Three blanks:
#
#       "error"          the status of a file that could not be read
#       None             an order ID that was not found
#       "needs review"   the status that sends a record to a person
# ============================================================

try:
    # BEGIN SOLUTION
    for row in requests:
        if "error" in row:
            row["status"], row["reason"] = "error", row["error"]
        elif row["order_id"] is None:
            row["status"], row["reason"] = "needs review", "no order number in the email"
        else:
            row["status"], row["reason"] = "ok", ""
    # SCAFFOLD: for row in requests:
    # SCAFFOLD:     if "error" in row:
    # SCAFFOLD:         row["status"], row["reason"] = ..., row["error"]   # "error"
    # SCAFFOLD:     elif row["order_id"] is ...:   # None
    # SCAFFOLD:         row["status"], row["reason"] = ..., "no order number in the email"   # "needs review"
    # SCAFFOLD:     else:
    # SCAFFOLD:         row["status"], row["reason"] = "ok", ""
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for row in requests:
    print(f"{row['email']}  {str(row.get('order_id')):14} {row['status']:13} {row['reason']}")

check("status_reason", requests=requests)

---

## Task 3 - Save safely and summarize

In [ ]:
# ============================================================
# TASK 3 - Save safely and summarize
# ============================================================
#
# WHAT TO DO
#   Write the batch to a temporary file, then move it over
#   return_requests.json in one step. Count the records by
#   status for the summary line.
#
# WHY IT MATTERS
#   Another system reads return_requests.json. Writing it in
#   place means that a crash halfway leaves half a file - and the
#   reader takes it as a smaller batch. Writing a temporary file
#   and replacing the old one in a single step means readers see
#   the old batch or the new one, never a broken one. The
#   summary is what the team looks at first.
#
# WHERE TO SEE IT IN THE LECTURE
#   "At the end, we save the list as a JSON file".
#
# HOW TO DO IT
#   Three blanks:
#
#       requests      the records to write
#       target        replace the real file with the temporary one
#       "status"      what the summary counts by
# ============================================================

target = Path("return_requests.json")
temporary = Path("return_requests.json.tmp")
try:
    # BEGIN SOLUTION
    temporary.write_text(json.dumps(requests, indent=2))
    temporary.replace(target)
    summary = Counter(row["status"] for row in requests)
    # SCAFFOLD: temporary.write_text(json.dumps(..., indent=2))   # requests
    # SCAFFOLD: temporary.replace(...)   # target
    # SCAFFOLD: summary = Counter(row[...] for row in requests)   # "status"
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError, KeyError) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
saved = json.loads(target.read_text())
print(len(saved), "records saved:", dict(summary))

check("save_summary", saved=saved, summary=summary)

---

## Done

Run the checked extraction over the whole batch, isolate each file's failure, keep a status and a reason per record, and write the output so that readers never see a broken file. Route the records with missing values to a person instead of forcing a value.